In [1]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
from tqdm import tqdm
from simple_rq_vae import SimpleRQVAE

class EmbeddingDataset(Dataset):
    """Простой датасет для embeddings"""
    def __init__(self, embeddings_path):
        # Загружаем embeddings (numpy array)
        self.embeddings = np.load(embeddings_path)
        self.dim = self.embeddings.shape[-1]

    def __getitem__(self, index):
        emb = self.embeddings[index]
        return torch.FloatTensor(emb), index

    def __len__(self):
        return len(self.embeddings)

In [2]:
def train_rqvae(model, dataloader, optimizer, device, epoch):
    """Одна эпоха обучения"""
    model.train()
    total_loss = 0
    total_recon_loss = 0
    total_rq_loss = 0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch}")

    for batch_idx, (data, _) in enumerate(pbar):
        data = data.to(device)

        optimizer.zero_grad()

        # Forward pass
        x_recon, rq_loss, indices, z_q = model(data)

        # Compute loss
        total_loss_batch, recon_loss = model.compute_loss(x_recon, data, rq_loss)

        # Backward pass
        total_loss_batch.backward()
        optimizer.step()

        # Accumulate losses
        total_loss += total_loss_batch.item()
        total_recon_loss += recon_loss.item()
        total_rq_loss += rq_loss.item()

        # Update progress bar
        pbar.set_postfix({
            'Loss': f'{total_loss_batch.item():.4f}',
            'Recon': f'{recon_loss.item():.4f}',
            'RQ': f'{rq_loss.item():.4f}'
        })

    avg_total_loss = total_loss / len(dataloader)
    avg_recon_loss = total_recon_loss / len(dataloader)
    avg_rq_loss = total_rq_loss / len(dataloader)

    return avg_total_loss, avg_recon_loss, avg_rq_loss

In [3]:
@torch.no_grad()
def evaluate_collision_rate(model, dataloader, device):
    """Вычисляем collision rate для семантических ID"""
    model.eval()

    all_semantic_ids = []

    for data, _ in dataloader:
        data = data.to(device)
        indices = model.get_semantic_ids(data)
        all_semantic_ids.extend(indices.cpu().numpy())

    # Считаем коллизии
    unique_ids = set()
    total_items = len(all_semantic_ids)

    for indices in all_semantic_ids:
        id_str = "-".join([str(idx) for idx in indices])
        unique_ids.add(id_str)

    collision_rate = (total_items - len(unique_ids)) / total_items
    return collision_rate

In [33]:
from dataset_utils import EmbDataset


def main():
    # Параметры
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    batch_size = 1024
    learning_rate = 1e-3
    epochs = 100

    # Создаем модель (настройки как в TIGER)
    model = SimpleRQVAE(
        in_dim=768,  # Sentence-T5 embeddings
        num_emb_list=[256, 256, 256],  # 3 levels, 256 codes each
        e_dim=32,  # latent dimension
        hidden_layers=[512, 256, 128],  # encoder/decoder layers
        beta=0.25,  # commitment loss weight
        loss_type="mse"
    ).to(device)

    print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")

    # Датасет и даталоадер
    # dataset = EmbeddingDataset("path/to/your/embeddings.npy")
    # dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4)



    dataset = EmbDataset("data/Beauty/_final_data.pkl")
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4)

    train_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    eval_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)  # для генерации ID

    # Optimizer
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

    print("Начинаем обучение...")
    best_collision_rate = float('inf')

    for epoch in range(epochs):
        # Обучение
        avg_total_loss, avg_recon_loss, avg_rq_loss = train_rqvae(
            model, train_dataloader, optimizer, device, epoch
        )

        print(f"Epoch {epoch}: Total Loss: {avg_total_loss:.4f}, "
              f"Recon Loss: {avg_recon_loss:.4f}, RQ Loss: {avg_rq_loss:.4f}")

        # Проверяем collision rate каждые 10 эпох
        if (epoch + 1) % 10 == 0:
            collision_rate = evaluate_collision_rate(model, eval_dataloader, device)
            print(f"Collision Rate: {collision_rate:.4f}")

            # Сохраняем лучшую модель
            if collision_rate < best_collision_rate:
                best_collision_rate = collision_rate
                torch.save(model.state_dict(), 'best_rqvae_model.pth')
                print(f"Saved best model with collision rate: {best_collision_rate:.4f}")

    print("\n" + "="*50)
    print("ОБУЧЕНИЕ ЗАВЕРШЕНО! Генерируем semantic IDs для датасета...")
    print("="*50)

    # Генерируем и сохраняем semantic IDs для всего датасета
    stats = generate_semantic_ids_for_dataset(
        model, eval_dataloader, device, save_dir="semantic_ids/"
    )

    print("\n" + "="*50)
    print("ГЕНЕРАЦИЯ SEMANTIC IDS ЗАВЕРШЕНА!")
    print("="*50)
    print("Файлы сохранены:")
    print("- semantic_ids/semantic_ids_3token.npy - основные 3-токенные ID")
    print("- semantic_ids/semantic_ids_4token.npy - уникальные 4-токенные ID")
    print("- semantic_ids/item_to_semantic_id.pkl - маппинг item -> 3-token ID")
    print("- semantic_ids/item_to_unique_id.pkl - маппинг item -> 4-token ID")
    print("- semantic_ids/semantic_ids_stats.json - статистика")
    print("- semantic_ids/semantic_ids_samples.json - примеры")


In [37]:
import pickle
import json
import os


@torch.no_grad()
def generate_semantic_ids_for_dataset(model, dataloader, device, save_dir="semantic_ids/"):
    """
    Генерирует semantic IDs для всего датасета и сохраняет в разных форматах
    как в оригинальной реализации LETTER
    """
    model.eval()

    # Создаем директорию для сохранения
    os.makedirs(save_dir, exist_ok=True)

    print("Генерируем semantic IDs для всего датасета...")

    all_semantic_ids = []
    all_unique_ids = []
    all_item_indices = []
    collision_counter = {}

    pbar = tqdm(dataloader, desc="Generating Semantic IDs")

    for batch_idx, (data, item_indices) in enumerate(pbar):
        data = data.to(device)

        # Получаем semantic IDs (3 токена)
        semantic_ids = model.get_semantic_ids(data)  # [batch_size, 3]

        # Обрабатываем каждый элемент в батче
        for i, (semantic_id, item_idx) in enumerate(zip(semantic_ids, item_indices)):
            semantic_id_tuple = tuple(semantic_id.cpu().numpy())

            # Формируем строковое представление для проверки коллизий
            id_str = "-".join([str(x) for x in semantic_id_tuple])

            # Считаем коллизии и добавляем четвертый токен
            if id_str not in collision_counter:
                collision_counter[id_str] = 0
                collision_token = 0
            else:
                collision_counter[id_str] += 1
                collision_token = collision_counter[id_str]

            # Создаем уникальный ID с четвертым токеном
            unique_id = semantic_id_tuple + (collision_token,)

            all_semantic_ids.append(semantic_id_tuple)  # 3 токена
            all_unique_ids.append(unique_id)  # 4 токена
            all_item_indices.append(item_idx.item())

    print(f"Сгенерировано {len(all_semantic_ids)} semantic IDs")
    print(f"Коллизий найдено: {sum(1 for v in collision_counter.values() if v > 0)}")

    # 1. Сохраняем в формате numpy (как в оригинале)
    semantic_ids_array = np.array(all_semantic_ids)  # [N, 3]
    unique_ids_array = np.array(all_unique_ids)  # [N, 4]

    np.save(os.path.join(save_dir, "semantic_ids_3token.npy"), semantic_ids_array)
    np.save(os.path.join(save_dir, "semantic_ids_4token.npy"), unique_ids_array)

    print(f"Сохранено в {save_dir}semantic_ids_3token.npy (форма: {semantic_ids_array.shape})")
    print(f"Сохранено в {save_dir}semantic_ids_4token.npy (форма: {unique_ids_array.shape})")

    # 2. Сохраняем маппинг item_index -> semantic_id (как в оригинале)
    item_to_semantic_id = {}
    item_to_unique_id = {}

    for item_idx, semantic_id, unique_id in zip(all_item_indices, all_semantic_ids, all_unique_ids):
        item_to_semantic_id[item_idx] = semantic_id
        item_to_unique_id[item_idx] = unique_id

    # Сохраняем как pickle (бинарный, быстрый)
    with open(os.path.join(save_dir, "item_to_semantic_id.pkl"), "wb") as f:
        pickle.dump(item_to_semantic_id, f)

    with open(os.path.join(save_dir, "item_to_unique_id.pkl"), "wb") as f:
        pickle.dump(item_to_unique_id, f)

    # Сохраняем как JSON (читаемый)
    item_to_semantic_id_json = {str(k): [int(v_x) for v_x in list(v)] for k, v in item_to_semantic_id.items()}
    item_to_unique_id_json = {str(k): [int(v_x) for v_x in list(v)] for k, v in item_to_unique_id.items()}

    with open(os.path.join(save_dir, "item_to_semantic_id.json"), "w") as f:
        print("bababa", [{str(k): [int(v_x) for v_x in list(v)] for k, v in item_to_semantic_id.items()}.items()][:10])
        json.dump(item_to_semantic_id_json, f, indent=2)

    with open(os.path.join(save_dir, "item_to_unique_id.json"), "w") as f:
        json.dump(item_to_unique_id_json, f, indent=2)

    print(f"Сохранен маппинг item_index -> semantic_id в {save_dir}")

    # 3. Сохраняем статистику (как в оригинале)
    stats = {
        "total_items": int(len(all_semantic_ids)),
        "unique_3token_ids": int(len(set(all_semantic_ids))),
        "unique_4token_ids": int(len(set(all_unique_ids))),
        "collision_rate_3token": float(1.0 - len(set(all_semantic_ids)) / len(all_semantic_ids)),
        "collision_rate_4token": float(1.0 - len(set(all_unique_ids)) / len(all_unique_ids)),
        "codebook_sizes": [int(x) for x in model.num_emb_list],  # ИСПРАВЛЕНО!
        "theoretical_capacity": int(np.prod(model.num_emb_list)),
        "codebook_usage": float(len(set(all_semantic_ids)) / np.prod(model.num_emb_list))
    }

    with open(os.path.join(save_dir, "semantic_ids_stats.json"), "w") as f:
        json.dump(stats, f, indent=2)

    print("\n=== Статистика Semantic IDs ===")
    print(f"Всего элементов: {stats['total_items']}")
    print(f"Уникальных 3-токенных ID: {stats['unique_3token_ids']}")
    print(f"Уникальных 4-токенных ID: {stats['unique_4token_ids']}")
    print(f"Collision rate (3 токена): {stats['collision_rate_3token']:.4f}")
    print(f"Collision rate (4 токена): {stats['collision_rate_4token']:.4f}")
    print(f"Использование codebook: {stats['codebook_usage']:.4f}")
    print(f"Теоретическая емкость: {stats['theoretical_capacity']}")

    # 4. Сохраняем примеры ID (для визуального контроля)
    sample_size = min(10, len(all_semantic_ids))
    samples = {
        "3token_examples": [[int(x) for x in all_semantic_ids[i]] for i in range(sample_size)],
        "4token_examples": [[int(x) for x in all_unique_ids[i]] for i in range(sample_size)],
        "item_indices": [int(all_item_indices[i]) for i in range(sample_size)]
    }

    with open(os.path.join(save_dir, "semantic_ids_samples.json"), "w") as f:
        json.dump(samples, f, indent=2)

    print(f"\nПримеры semantic IDs сохранены в {save_dir}semantic_ids_samples.json")

    return stats

In [38]:
def load_semantic_ids(save_dir="semantic_ids/"):
    """
    Вспомогательная функция для загрузки сгенерированных semantic IDs
    """
    # Загружаем массивы
    semantic_ids_3token = np.load(os.path.join(save_dir, "semantic_ids_3token.npy"))
    semantic_ids_4token = np.load(os.path.join(save_dir, "semantic_ids_4token.npy"))

    # Загружаем маппинги
    with open(os.path.join(save_dir, "item_to_semantic_id.pkl"), "rb") as f:
        item_to_semantic_id = pickle.load(f)

    with open(os.path.join(save_dir, "item_to_unique_id.pkl"), "rb") as f:
        item_to_unique_id = pickle.load(f)

    # Загружаем статистику
    with open(os.path.join(save_dir, "semantic_ids_stats.json"), "r") as f:
        stats = json.load(f)

    print(f"Загружены semantic IDs:")
    print(f"- 3-токенные: {semantic_ids_3token.shape}")
    print(f"- 4-токенные: {semantic_ids_4token.shape}")
    print(f"- Collision rate: {stats['collision_rate_3token']:.4f}")

    return {
        "semantic_ids_3token": semantic_ids_3token,
        "semantic_ids_4token": semantic_ids_4token,
        "item_to_semantic_id": item_to_semantic_id,
        "item_to_unique_id": item_to_unique_id,
        "stats": stats
    }

if __name__ == "__main__":
    main()

    # Пример загрузки результатов
    print("\nПример загрузки semantic IDs:")
    try:
        data = load_semantic_ids()
        print("Успешно загружено!")

        # Показываем несколько примеров
        print("\nПримеры semantic IDs:")
        for i in range(5):
            print(f"Item {i}: 3-token={data['semantic_ids_3token'][i]}, 4-token={data['semantic_ids_4token'][i]}")

    except FileNotFoundError:
        print("Файлы semantic IDs не найдены. Сначала запустите обучение.")

Model parameters: 1149472
Max item_id: 12101
Number of unique items in mapping: 12101
Начинаем обучение...


Epoch 0: 100%|██████████| 12/12 [00:00<00:00, 58.87it/s, Loss=0.0005, Recon=0.0001, RQ=0.0004]


Epoch 0: Total Loss: 0.0004, Recon Loss: 0.0003, RQ Loss: 0.0002


Epoch 1: 100%|██████████| 12/12 [00:00<00:00, 59.24it/s, Loss=0.0002, Recon=0.0001, RQ=0.0001]


Epoch 1: Total Loss: 0.0006, Recon Loss: 0.0001, RQ Loss: 0.0005


...

Epoch 98: Total Loss: 0.0000, Recon Loss: 0.0000, RQ Loss: 0.0000


Epoch 99: 100%|██████████| 12/12 [00:00<00:00, 65.24it/s, Loss=0.0000, Recon=0.0000, RQ=0.0000]


Epoch 99: Total Loss: 0.0000, Recon Loss: 0.0000, RQ Loss: 0.0000
Collision Rate: 0.6791

ОБУЧЕНИЕ ЗАВЕРШЕНО! Генерируем semantic IDs для датасета...
Генерируем semantic IDs для всего датасета...


Generating Semantic IDs: 100%|██████████| 12/12 [00:00<00:00, 148.60it/s]


Сгенерировано 12101 semantic IDs
Коллизий найдено: 1348
Сохранено в semantic_ids/semantic_ids_3token.npy (форма: (12101, 3))
Сохранено в semantic_ids/semantic_ids_4token.npy (форма: (12101, 4))
bababa [dict_items([('0', [226, 29, 253]), ('1', [80, 178, 61]), ('2', [83, 228, 25]), ('3', [45, 132, 61]), ('4', [93, 184, 104]), ('5', [118, 217, 104]), ('6', [83, 154, 252]), ('7', [150, 246, 104]), ('8', [202, 136, 202]), ('9', [159, 154, 252]), ('10', [226, 79, 104]), ('11', [114, 73, 61]), ('12', [75, 27, 61]), ('13', [80, 184, 61]), ('14', [114, 27, 61]), ('15', [45, 99, 61]), ('16', [61, 244, 243]), ('17', [118, 181, 195]), ('18', [114, 98, 61]), ('19', [93, 152, 61]), ('20', [17, 217, 104]), ('21', [128, 76, 61]), ('22', [118, 228, 104]), ('23', [80, 129, 104]), ('24', [122, 166, 61]), ('25', [150, 156, 203]), ('26', [150, 57, 180]), ('27', [118, 181, 61]), ('28', [114, 129, 61]), ('29', [17, 189, 104]), ('30', [45, 138, 104]), ('31', [114, 76, 104]), ('32', [118, 189, 73]), ('33', [15

In [35]:
def fix_json_conversion():
    """
    Демонстрация правильной конвертации для JSON
    """
    import numpy as np

    # Проблемная ситуация - tuple с numpy int64
    semantic_id_tuple = (np.int64(207), np.int64(249), np.int64(104))
    print(f"Проблемный tuple: {semantic_id_tuple}")
    print(f"Типы элементов: {[type(x) for x in semantic_id_tuple]}")

    # Неправильное исправление - list() не конвертирует типы элементов
    wrong_list = list(semantic_id_tuple)
    print(f"list() - НЕ исправляет типы: {[type(x) for x in wrong_list]}")

    # Правильное исправление - явная конвертация каждого элемента
    correct_list = [int(x) for x in semantic_id_tuple]
    print(f"[int(x) for x in ...] - исправляет типы: {[type(x) for x in correct_list]}")

    # Тест JSON сериализации
    import json

    try:
        json.dumps(wrong_list)
        print("❌ Неожиданно: wrong_list сериализовался")
    except TypeError as e:
        print(f"✅ Ожидаемо: wrong_list не сериализуется - {e}")

    try:
        result = json.dumps(correct_list)
        print(f"✅ Правильно: correct_list сериализуется - {result}")
    except TypeError as e:
        print(f"❌ Неожиданно: correct_list не сериализуется - {e}")

fix_json_conversion()

Проблемный tuple: (207, 249, 104)
Типы элементов: [<class 'numpy.int64'>, <class 'numpy.int64'>, <class 'numpy.int64'>]
list() - НЕ исправляет типы: [<class 'numpy.int64'>, <class 'numpy.int64'>, <class 'numpy.int64'>]
[int(x) for x in ...] - исправляет типы: [<class 'int'>, <class 'int'>, <class 'int'>]
✅ Ожидаемо: wrong_list не сериализуется - Object of type int64 is not JSON serializable
✅ Правильно: correct_list сериализуется - [207, 249, 104]
